# Data Validation - Duplicates and Nulls on Keys
Validation checks for dimensional and fact tables to ensure data integrity on primary and composite keys.

In [15]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Get project root
src_path = Path.cwd()
project_src = src_path.parents[1]
print(f"Project source root: {project_src}")

Project source root: c:\Users\davib\Desktop\MSc_DataScience\DataWarehouse\lab_assignment\imbd


## Load All Tables

In [ ]:
# Load all silver layer parquet files
# ── Colleague's original tables ──────────────────────────
dim_profession          = pd.read_parquet(f'{project_src}\\data\\silver\\dim_profession.parquet')
bridge_profession_group = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet')
dim_person              = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
dim_roles               = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet')
dim_title_basic         = pd.read_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet')
bridge_kwn_titles       = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet')
participations_pers     = pd.read_parquet(f'{project_src}\\data\\silver\\participations_pers.parquet')

# ── Renamed: dim_genres → dim_genre, bridge_genres → bridge_genre_group ──
dim_genre           = pd.read_parquet(f'{project_src}\\data\\silver\\dim_genre.parquet')
bridge_genre_group  = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_genre_group.parquet')

# ── New TV series tables ──────────────────────────────────
dim_time                = pd.read_parquet(f'{project_src}\\data\\silver\\dim_time.parquet')
dim_series              = pd.read_parquet(f'{project_src}\\data\\silver\\dim_series.parquet')
dim_season              = pd.read_parquet(f'{project_src}\\data\\silver\\dim_season.parquet')
fact_series_performance = pd.read_parquet(f'{project_src}\\data\\silver\\fact_series_performance.parquet')

print("✓ All tables loaded successfully")

✓ All tables loaded successfully


## Validation Functions

In [16]:
def check_null_keys(df, table_name, key_columns):
    """Check for null values in key columns"""
    print(f"\n{'='*60}")
    print(f"NULL CHECK: {table_name}")
    print(f"{'='*60}")
    
    null_found = False
    for col in key_columns:
        null_count = df[col].isna().sum()
        if null_count > 0:
            print(f"  ✗ {col}: {null_count} NULL values found")
            null_found = True
        else:
            print(f"  ✓ {col}: No NULL values")
    
    if not null_found:
        print(f"  ✓ All key columns are NULL-free")
    return not null_found

def check_duplicate_keys(df, table_name, key_columns):
    """Check for duplicate values in key columns"""
    print(f"\n{'='*60}")
    print(f"DUPLICATE CHECK: {table_name}")
    print(f"{'='*60}")
    
    duplicates = df.duplicated(subset=key_columns, keep=False).sum()
    
    if duplicates > 0:
        print(f"  ✗ Found {duplicates} duplicate key combinations")
        print(f"\n  Duplicate rows:")
        dup_rows = df[df.duplicated(subset=key_columns, keep=False)].sort_values(key_columns)
        print(dup_rows[key_columns].to_string())
        return False
    else:
        print(f"  ✓ No duplicate keys found")
        return True

def validate_table(df, table_name, key_columns):
    """Comprehensive validation for a table"""
    print(f"\n{'#'*60}")
    print(f"# VALIDATING: {table_name}")
    print(f"# Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"# Keys: {', '.join(key_columns)}")
    print(f"{'#'*60}")
    
    null_valid = check_null_keys(df, table_name, key_columns)
    dup_valid = check_duplicate_keys(df, table_name, key_columns)
    
    overall = "✓ PASSED" if (null_valid and dup_valid) else "✗ FAILED"
    print(f"\n{overall}\n")
    return null_valid and dup_valid

# Dimensional Tables Validation

In [4]:
## dim_profession
validate_table(dim_profession, "dim_profession", ['profession_id'])


############################################################
# VALIDATING: dim_profession
# Shape: 41 rows × 2 columns
# Keys: profession_id
############################################################

NULL CHECK: dim_profession
  ✓ profession_id: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: dim_profession
  ✓ No duplicate keys found

✓ PASSED



True

In [5]:
## dim_person
validate_table(dim_person, "dim_person", ['nconst'])


############################################################
# VALIDATING: dim_person
# Shape: 10777803 rows × 5 columns
# Keys: nconst
############################################################

NULL CHECK: dim_person
  ✓ nconst: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: dim_person
  ✓ No duplicate keys found

✓ PASSED



True

In [6]:
## dim_roles
validate_table(dim_roles, "dim_roles", ['role_id'])


############################################################
# VALIDATING: dim_roles
# Shape: 43512883 rows × 7 columns
# Keys: role_id
############################################################

NULL CHECK: dim_roles
  ✓ role_id: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: dim_roles
  ✓ No duplicate keys found

✓ PASSED



True

In [7]:
## dim_genre
validate_table(dim_genre, "dim_genre", ['genre_id'])


############################################################
# VALIDATING: dim_genres
# Shape: 21 rows × 2 columns
# Keys: genre_id
############################################################

NULL CHECK: dim_genres
  ✓ genre_id: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: dim_genres
  ✓ No duplicate keys found

✓ PASSED



True

In [8]:
## dim_title_basic
validate_table(dim_title_basic, "dim_title_basic", ['tconst'])


############################################################
# VALIDATING: dim_title_basic
# Shape: 1458 rows × 9 columns
# Keys: tconst
############################################################

NULL CHECK: dim_title_basic
  ✓ tconst: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: dim_title_basic
  ✓ No duplicate keys found

✓ PASSED



True

In [ ]:
## dim_time
validate_table(dim_time, "dim_time", ['year'])

In [ ]:
## dim_series
validate_table(dim_series, "dim_series", ['tconst'])

In [ ]:
## dim_season
validate_table(dim_season, "dim_season", ['season_id'])

# Bridge Tables Validation

In [9]:
## bridge_profession_group
validate_table(bridge_profession_group, "bridge_profession_group", ['nconst', 'profession_id'])


############################################################
# VALIDATING: bridge_profession_group
# Shape: 11721254 rows × 4 columns
# Keys: nconst, profession_id
############################################################

NULL CHECK: bridge_profession_group
  ✓ nconst: No NULL values
  ✓ profession_id: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: bridge_profession_group
  ✓ No duplicate keys found

✓ PASSED



True

In [10]:
## bridge_genre_group
validate_table(bridge_genre_group, "bridge_genre_group", ['tconst', 'genre_id'])


############################################################
# VALIDATING: bridge_genres
# Shape: 1616 rows × 4 columns
# Keys: tconst, genre_id
############################################################

NULL CHECK: bridge_genres
  ✓ tconst: No NULL values
  ✓ genre_id: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: bridge_genres
  ✓ No duplicate keys found

✓ PASSED



True

In [11]:
## bridge_kwn_titles
validate_table(bridge_kwn_titles, "bridge_kwn_titles", ['tconst', 'nconst'])


############################################################
# VALIDATING: bridge_kwn_titles
# Shape: 16851455 rows × 4 columns
# Keys: tconst, nconst
############################################################

NULL CHECK: bridge_kwn_titles
  ✓ tconst: No NULL values
  ✓ nconst: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: bridge_kwn_titles
  ✓ No duplicate keys found

✓ PASSED



True

# Fact Table Validation

In [18]:
## participations_pers
# Fact table: check for natural key combination (nconst, tconst, category, job)
validate_table(participations_pers, "participations_pers", ['nconst', 'tconst', 'role_id'])


############################################################
# VALIDATING: participations_pers
# Shape: 11672 rows × 14 columns
# Keys: nconst, tconst, role_id
############################################################

NULL CHECK: participations_pers
  ✗ nconst: 57 NULL values found
  ✓ tconst: No NULL values
  ✗ role_id: 57 NULL values found

DUPLICATE CHECK: participations_pers
  ✓ No duplicate keys found

✗ FAILED



False

In [ ]:
## fact_series_performance
validate_table(fact_series_performance, "fact_series_performance",
               ['tconst_series', 'season_id', 'release_year', 'genre_id'])

# Validation Summary

In [13]:
print("\n" + "="*60)
print("VALIDATION SUMMARY")
print("="*60)

validation_results = {
    "dim_time":                validate_table(dim_time,                "dim_time",                ['year']),
    "dim_series":              validate_table(dim_series,              "dim_series",              ['tconst']),
    "dim_season":              validate_table(dim_season,              "dim_season",              ['season_id']),
    "dim_genre":               validate_table(dim_genre,               "dim_genre",               ['genre_id']),
    "dim_profession":          validate_table(dim_profession,          "dim_profession",          ['profession_id']),
    "dim_person":              validate_table(dim_person,              "dim_person",              ['nconst']),
    "dim_roles":               validate_table(dim_roles,               "dim_roles",               ['role_id']),
    "dim_title_basic":         validate_table(dim_title_basic,         "dim_title_basic",         ['tconst']),
    "bridge_genre_group":      validate_table(bridge_genre_group,      "bridge_genre_group",      ['tconst', 'genre_id']),
    "bridge_profession_group": validate_table(bridge_profession_group, "bridge_profession_group", ['nconst', 'profession_id']),
    "bridge_kwn_titles":       validate_table(bridge_kwn_titles,       "bridge_kwn_titles",       ['tconst', 'nconst']),
    "participations_pers":     validate_table(participations_pers,     "participations_pers",     ['nconst', 'tconst', 'role_id']),
    "fact_series_performance": validate_table(fact_series_performance, "fact_series_performance", ['tconst_series', 'season_id', 'release_year', 'genre_id']),
}

total_tests  = len(validation_results)
passed_tests = sum(validation_results.values())
failed_tests = total_tests - passed_tests

print("\n" + "="*60)
print(f"OVERALL RESULTS: {passed_tests}/{total_tests} tables passed validation")
print("="*60)

for table, result in validation_results.items():
    status = "✓ PASS" if result else "✗ FAIL"
    print(f"{status}: {table}")


VALIDATION SUMMARY

############################################################
# VALIDATING: dim_profession
# Shape: 41 rows × 2 columns
# Keys: profession_id
############################################################

NULL CHECK: dim_profession
  ✓ profession_id: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: dim_profession
  ✓ No duplicate keys found

✓ PASSED


############################################################
# VALIDATING: dim_person
# Shape: 10777803 rows × 5 columns
# Keys: nconst
############################################################

NULL CHECK: dim_person
  ✓ nconst: No NULL values
  ✓ All key columns are NULL-free

DUPLICATE CHECK: dim_person
  ✓ No duplicate keys found

✓ PASSED


############################################################
# VALIDATING: dim_roles
# Shape: 43512883 rows × 7 columns
# Keys: role_id
############################################################

NULL CHECK: dim_roles
  ✓ role_id: No NULL values
  ✓ All k